In [ ]:
# ---------- Remove empty class folders (safe) ----------
from pathlib import Path
import shutil, os
import zipfile # Added import for zipfile

data_root = Path("/content/exp12_split")   # your dataset root
parts = ["train", "validate", "val", "test"]   # try common names
valid_exts = {".jpg",".jpeg",".png",".ppm",".bmp",".pgm",".tif",".tiff",".webp"}

# Unzip dataset if not already extracted
zip_path = Path("/content/exp12_split_r.zip")
if not data_root.exists():
    print(f"Extracting {zip_path.name} to {data_root}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_root)
    print("Extraction complete.")
    # After initial extraction, check for nested directory with the same name
    # e.g., if zip extracts to /content/exp12_split/exp12_split/...
    if (data_root / data_root.name).is_dir():
        data_root = data_root / data_root.name
        print(f"Adjusted data_root to: {data_root} due to nested extraction.")
else:
    print(f"Directory {data_root} already exists. Skipping extraction.")

# Robust check for nested structure, useful even if data_root existed before this run
# Check if the 'train' folder (or other main parts) exists directly under data_root
# If not, assume it's one level deeper, e.g., /content/exp12_split/exp12_split/train
if not (data_root / "train").exists() and (data_root / data_root.name / "train").exists():
    data_root = data_root / data_root.name
    print(f"Further adjusted data_root to: {data_root} because 'train' was found nested.")

def remove_empty_class_folders(root):
    root = Path(root)
    for part in parts:
        p = root/part
        if not p.exists():
            continue
        for cls in sorted(p.iterdir()):
            if not cls.is_dir():
                continue
            total_files = sum(1 for f in cls.rglob("*") if f.is_file())
            if total_files == 0:
                print(f"Removing empty class folder: {cls}")
                shutil.rmtree(cls)
            else:
                ext_count = sum(1 for f in cls.rglob("*") if f.suffix.lower() in valid_exts)
                print(f"Keeping {cls.name}: total_files={total_files}, ext_matches={ext_count}")

remove_empty_class_folders(data_root)

# Now you can safely use ImageFolder as before:
from torchvision.datasets import ImageFolder
import torchvision.transforms as T # Added import for T
img_size = 128 # Added img_size definition from cell jYV-HvoHH2m9
transform = T.Compose([ # Added transform definition from cell jYV-HvoHH2m9
    T.Resize((img_size, img_size)),
    T.ToTensor(),              # scales to [0,1]
])
train_ds = ImageFolder(data_root/"train", transform=transform)
val_ds   = ImageFolder(data_root/"validate", transform=transform) if (data_root/"validate").exists() else ImageFolder(data_root/"val", transform=transform)
test_ds  = ImageFolder(data_root/"test", transform=transform)


In [ ]:
# Exp12_autoencoder.py
# Requirements: torch, torchvision, matplotlib, scikit-image, tqdm
# Unzip dataset, build dataloaders, train convolutional autoencoder, evaluate with MSE/PSNR/SSIM and show curves.

import os
import zipfile
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid, save_image

from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim

# ========== 1) Unzip dataset (if not already unzipped) ==========
from pathlib import Path
from torchvision.datasets.folder import default_loader
from torch.utils.data import Dataset
from PIL import Image

class SkipEmptyClassImageFolder(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        classes = sorted([d for d in self.root.iterdir() if d.is_dir()]) # Corrected 'd d' to 'd'
        self.samples = []
        self.class_to_idx = {}
        idx = 0
        for c in classes:
            files = [f for f in c.rglob("*") if f.is_file()]
            # keep only image-like by extension
            img_files = [f for f in files if f.suffix.lower() in valid_exts]
            if len(img_files) == 0:
                print(f"Skipping empty class: {c.name}")
                continue
            self.class_to_idx[c.name] = idx
            for f in img_files:
                self.samples.append((str(f), idx))
            idx += 1
        self.transform = transform

    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

# Usage:
train_ds = SkipEmptyClassImageFolder(data_root/"train", transform=transform)
val_ds   = SkipEmptyClassImageFolder(data_root/"val", transform=transform)
test_ds  = SkipEmptyClassImageFolder(data_root/"test", transform=transform)


# ========== 2) Hyperparameters & transforms ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 128
batch_size = 64
num_workers = 4

transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),              # scales to [0,1]
])

train_ds = ImageFolder(data_root/"train", transform=transform)
val_ds   = ImageFolder(data_root/"val", transform=transform)
test_ds  = ImageFolder(data_root/"test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

# ========== 3) Model: simple Conv Autoencoder ==========
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        # Encoder
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),  # 64x64
            nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1), # 32x32
            nn.ReLU(True),
            nn.Conv2d(64, 128, 4, 2, 1),# 16x16
            nn.ReLU(True),
            nn.Conv2d(128, 256, 4, 2, 1),#8x8
            nn.ReLU(True),
            nn.Flatten(),
            nn.Linear(256*8*8, latent_dim),
            nn.ReLU(True)
        )
        # Decoder
        self.dec_fc = nn.Linear(latent_dim, 256*8*8)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), #16
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  #32
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   #64
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),    #128
            nn.Sigmoid()  # output in [0,1]
        )

    def forward(self, x):
        z = self.enc(x)
        x_ = self.dec_fc(z).view(-1, 256, 8, 8)
        x_rec = self.dec(x_)
        return x_rec, z

model = ConvAutoencoder(latent_dim=512).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# ========== 4) Training loop with tracking ==========
num_epochs = 20
train_losses, val_losses = [], []

# Create the directory /mnt/data if it doesn't exist
os.makedirs("/mnt/data", exist_ok=True)

for epoch in range(1, num_epochs+1):
    model.train()
    running_loss = 0.0
    for imgs, _ in tqdm(train_loader, desc=f"Train Epoch {epoch}/{num_epochs}"):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        rec, _ = model(imgs)
        loss = criterion(rec, imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            rec, _ = model(imgs)
            loss = criterion(rec, imgs)
            running_val += loss.item() * imgs.size(0)
    epoch_val_loss = running_val / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch}: train_loss={epoch_train_loss:.6f}, val_loss={epoch_val_loss:.6f}")

# Save model
torch.save(model.state_dict(), "/mnt/data/autoencoder_exp12.pth")
print("Model saved to /mnt/data/autoencoder_exp12.pth")

# ========== 5) Plot training curves ==========
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.title("Autoencoder Training & Validation Loss")
plt.grid(True)
plt.savefig("/mnt/data/exp12_loss_curve.png")
plt.show()
print("Saved loss curve -> /mnt/data/exp12_loss_curve.png")

# ========== 6) Evaluate on test set: MSE, PSNR, SSIM ==========
model.eval()
mse_list, psnr_list, ssim_list = [], [], []
recon_examples = []
with torch.no_grad():
    for i, (img, _) in enumerate(tqdm(test_loader, desc="Evaluate test set")):
        img = img.to(device)
        rec, _ = model(img)
        # move to cpu numpy
        img_np = img.squeeze(0).permute(1,2,0).cpu().numpy()
        rec_np = rec.squeeze(0).permute(1,2,0).cpu().numpy()
        mse = np.mean((img_np - rec_np)**2)
        mse_list.append(mse)
        # PSNR expects data range 1.0
        psnr = compare_psnr(img_np, rec_np, data_range=1.0)
        psnr_list.append(psnr)
        # SSIM wants grayscale or multichannel=True
        ssim = compare_ssim(img_np, rec_np, data_range=1.0, win_size=7, channel_axis=-1)
        ssim_list.append(ssim)
        if i < 8:  # save a few examples
            recon_examples.append((img_np, rec_np))
    # break removed to evaluate all

print(f"Test MSE: {np.mean(mse_list):.6f}, PSNR: {np.mean(psnr_list):.3f}, SSIM: {np.mean(ssim_list):.4f}")

# ========== 7) Show and save reconstruction examples ==========
def show_examples(examples, ncols=4):
    n = len(examples)
    fig, axes = plt.subplots(2, ncols, figsize=(ncols*3, 2*3))
    for i, (orig, rec) in enumerate(examples):
        r, c = divmod(i, ncols)
        ax1 = axes[0, c]
        ax2 = axes[1, c]
        ax1.imshow(np.clip(orig,0,1))
        ax1.axis("off")
        ax1.set_title("Original")
        ax2.imshow(np.clip(rec,0,1))
        ax2.axis("off")
        ax2.set_title("Reconstruction")
    plt.tight_layout()
    plt.savefig("/mnt/data/exp12_recon_examples.png")
    plt.show()

show_examples(recon_examples)
print("Saved recon examples -> /mnt/data/exp12_recon_examples.png")

# ========== 8) Notes on metrics ==========
# - MSE is the optimization target (smaller is better).
# - PSNR higher is better (in dB).
# - SSIM in [0,1] with higher = better perceptual quality.
